In [ ]:
!rm -rf /kaggle/tmp/fred
!git clone -q --depth 1 --branch kaggle-shards-v2 \
  https://github.com/trungdangtapcode/FRED-Video-Search-System-Resumable.git \
  /kaggle/tmp/fred
%cd /kaggle/tmp/fred

In [ ]:
SHARD = "L21/part-001.json"
INPUT = "/kaggle/input/datasets/truonghaha/aic-2026-dataset"
DATA = "/kaggle/working/data"

!python -m keyframe_extraction \
  --input {INPUT} \
  --manifest manifests/aic-2026/{SHARD} \
  --output {DATA}/extracted_keyframes \
  --metadata-dir {DATA}/frame_metadata \
  --status-dir {DATA}/extraction_status \
  --workers 4 \
  --no-merge

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path(f"manifests/aic-2026/{SHARD}").read_text())
SHARD_ID = manifest["shard_id"]
expected = len(manifest["videos"])
data = Path(DATA)
done = list((data / "extraction_status").glob("*.done.json"))
assert len(done) == expected, f"Missed video: {len(done)}/{expected}"

!rm -rf {DATA}/extracted_keyframes/.backup \
  {DATA}/extracted_keyframes/.partial \
  {DATA}/extraction_status/.locks
!tar -C /kaggle/working -cf /kaggle/tmp/{SHARD_ID}.tar data
!tar -tf /kaggle/tmp/{SHARD_ID}.tar > /dev/null
!rm -rf {DATA}
!mv /kaggle/tmp/{SHARD_ID}.tar /kaggle/working/
!sha256sum /kaggle/working/{SHARD_ID}.tar > /kaggle/working/{SHARD_ID}.tar.sha256
print("DONE:", SHARD_ID)